# 05 — Export for Power BI
Consolidates all processed data into clean CSVs + Excel for Power BI import

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

matches = pd.read_csv('../data/processed/matches_clean.csv')
deliveries = pd.read_csv('../data/processed/deliveries_clean.csv')
batter_stats = pd.read_csv('../data/processed/batter_stats.csv')
bowler_stats = pd.read_csv('../data/processed/bowler_stats.csv')
team_summary = pd.read_csv('../data/processed/team_season_summary.csv')
venue_stats = pd.read_csv('../data/processed/venue_stats.csv')
print('All files loaded.')

All files loaded.


## 1. Season Summary Table

In [2]:
valid = matches[matches['winner'] != 'No Result']

season_summary = valid.groupby('season').agg(
    total_matches=('id', 'count'),
    unique_teams=('team1', lambda x: pd.concat([x, valid.loc[x.index, 'team2']]).nunique()),
    venues=('venue', 'nunique'),
    most_wins=('winner', lambda x: x.value_counts().idxmax()),
    toss_win_rate=('toss_winner_won', 'mean')
).reset_index()
season_summary['toss_win_rate'] = (season_summary['toss_win_rate'] * 100).round(1)

# Avg first innings score per season
avg_fi = deliveries[deliveries['inning'] == 1].groupby(['match_id','season'])['total_runs'].sum().groupby('season').mean().round(1).reset_index()
avg_fi.columns = ['season', 'avg_first_innings']
season_summary = season_summary.merge(avg_fi, on='season', how='left')

season_summary.to_csv('../data/processed/season_summary.csv', index=False)
print('season_summary.csv saved')
print(season_summary.to_string(index=False))

season_summary.csv saved
 season  total_matches  unique_teams  venues             most_wins  toss_win_rate  avg_first_innings
   2008             58             8       9      Rajasthan Royals           48.3              161.0
   2009             57             8       8        Delhi Capitals           57.9              150.3
   2010             60             8      13        Mumbai Indians           51.7              164.8
   2011             72            10      12   Chennai Super Kings           52.8              152.4
   2012             74             9      12 Kolkata Knight Riders           44.6              157.5
   2013             76             9      13        Mumbai Indians           47.4              155.9
   2014             60             8      13          Punjab Kings           50.0              163.1
   2015             57             8      13   Chennai Super Kings           49.1              166.3
   2016             60             8      11   Sunrisers Hyderabad

## 2. Match-level Fact Table (for Power BI relationships)

In [3]:
match_fact = matches[[
    'id','season','date','venue','city',
    'team1','team2','toss_winner','toss_decision','toss_winner_won',
    'winner','result','result_margin','method',
    'player_of_match','match_num'
]].copy()
match_fact.to_csv('../data/processed/match_fact.csv', index=False)
print(f'match_fact.csv saved — {len(match_fact)} rows')

match_fact.csv saved — 1095 rows


## 3. Ball-by-Ball Aggregated (over-level for Power BI — ball level too large)

In [4]:
over_agg = deliveries.groupby(['match_id','inning','over','batting_team','bowling_team','season','phase']).agg(
    runs=('total_runs', 'sum'),
    wickets=('is_wicket', 'sum'),
    fours=('is_four', 'sum'),
    sixes=('is_six', 'sum'),
    dots=('is_dot', 'sum'),
    balls=('ball', 'count')
).reset_index()
over_agg['run_rate'] = (over_agg['runs'] / over_agg['balls'] * 6).round(2)
over_agg.to_csv('../data/processed/over_summary.csv', index=False)
print(f'over_summary.csv saved — {len(over_agg)} rows')

over_summary.csv saved — 42210 rows


## 4. Export All to Excel (one file, multiple sheets)

In [5]:
output_path = '../data/processed/IPL_PowerBI_Data.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    match_fact.to_excel(writer, sheet_name='Matches', index=False)
    team_summary.to_excel(writer, sheet_name='Team_Season', index=False)
    batter_stats.to_excel(writer, sheet_name='Batters', index=False)
    bowler_stats.to_excel(writer, sheet_name='Bowlers', index=False)
    venue_stats.to_excel(writer, sheet_name='Venues', index=False)
    season_summary.to_excel(writer, sheet_name='Seasons', index=False)
    over_agg.to_excel(writer, sheet_name='Over_Summary', index=False)

print(f'IPL_PowerBI_Data.xlsx saved with 7 sheets!')
print('Sheets: Matches | Team_Season | Batters | Bowlers | Venues | Seasons | Over_Summary')

IPL_PowerBI_Data.xlsx saved with 7 sheets!
Sheets: Matches | Team_Season | Batters | Bowlers | Venues | Seasons | Over_Summary


## 5. Power BI Setup Guide

In [6]:
guide = """
=== POWER BI IMPORT GUIDE ===

1. Open Power BI Desktop
2. Get Data → Excel → select IPL_PowerBI_Data.xlsx
3. Load all 7 sheets

=== RELATIONSHIPS TO SET ===
Matches[id]           → Over_Summary[match_id]   (1:Many)
Matches[season]       → Seasons[season]           (Many:1)
Matches[season]       → Team_Season[season]       (Many:Many)
Matches[venue]        → Venues[venue]             (Many:1)

=== KEY MEASURES TO CREATE ===
Win % = DIVIDE(SUM(Team_Season[won]), SUM(Team_Season[played]))
Avg Score = AVERAGE(Matches[target_runs])
Toss Win Rate = AVERAGE(Matches[toss_winner_won])

=== SUGGESTED PAGES ===
Page 1 — Overview KPIs (cards: total matches, total runs, total sixes, seasons)
Page 2 — Teams (bar: win %, line: season wins, matrix: H2H)
Page 3 — Batsmen (table: top scorers, scatter: SR vs Avg)
Page 4 — Bowlers (table: wickets+economy, bar: dot %)
Page 5 — Venues (map + bar: avg score, chase %)
"""
print(guide)
with open('../powerbi/POWERBI_GUIDE.txt', 'w') as f:
    f.write(guide)


=== POWER BI IMPORT GUIDE ===

1. Open Power BI Desktop
2. Get Data → Excel → select IPL_PowerBI_Data.xlsx
3. Load all 7 sheets

=== RELATIONSHIPS TO SET ===
Matches[id]           → Over_Summary[match_id]   (1:Many)
Matches[season]       → Seasons[season]           (Many:1)
Matches[season]       → Team_Season[season]       (Many:Many)
Matches[venue]        → Venues[venue]             (Many:1)

=== KEY MEASURES TO CREATE ===
Win % = DIVIDE(SUM(Team_Season[won]), SUM(Team_Season[played]))
Avg Score = AVERAGE(Matches[target_runs])
Toss Win Rate = AVERAGE(Matches[toss_winner_won])

=== SUGGESTED PAGES ===
Page 1 — Overview KPIs (cards: total matches, total runs, total sixes, seasons)
Page 2 — Teams (bar: win %, line: season wins, matrix: H2H)
Page 3 — Batsmen (table: top scorers, scatter: SR vs Avg)
Page 4 — Bowlers (table: wickets+economy, bar: dot %)
Page 5 — Venues (map + bar: avg score, chase %)

